In [93]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import least_squares

In [94]:
swaption_data = pd.read_csv("../data/processed/swaption_data_clean.csv")

In [95]:
swaption_data.head()

,Expiry,Tenor,Forward,Annuity,Offset,Straddle,Strangle,RiskReversal,Payer,Receiver,Strike_Payer,Strike_Receiver,T_expiry
0,3m,1y,0.0342,0.9581,0.0025,20,10,-2,4.0000,6.0000,0.0367,0.0317,0.2500
1,6m,1y,0.0333,0.9506,0.0050,31,13,-3,5.0000,8.0000,0.0383,0.0283,0.5000
2,1y,1y,0.0332,0.9350,0.0100,50,18,-3,7.5000,10.5000,0.0432,0.0232,1.0000
3,2y,1y,0.0350,0.9044,0.0100,78,42,1,21.5000,20.5000,0.0450,0.0250,2.0000
4,3y,1y,0.0366,0.9647,0.0150,96,46,3,24.5000,21.5000,0.0516,0.0216,3.0000


In [96]:
swaption_data["Tenor"] = swaption_data["Tenor"].str.replace("m", "").str.replace("y", "")
swaption_data["Tenor"] = swaption_data["Tenor"].astype(float)

In [97]:
swaption_data['Straddle'] = swaption_data['Straddle'] / 10000
swaption_data['Strangle'] = swaption_data['Strangle'] / 10000
swaption_data['RiskReversal'] = swaption_data['RiskReversal'] / 10000
swaption_data['Payer'] = swaption_data['Payer'] / 10000
swaption_data['Receiver'] = swaption_data['Receiver'] / 10000

In [98]:
swaption_data.head()

,Expiry,Tenor,Forward,Annuity,Offset,Straddle,Strangle,RiskReversal,Payer,Receiver,Strike_Payer,Strike_Receiver,T_expiry
0,3m,1.0000,0.0342,0.9581,0.0025,0.0020,0.0010,-0.0002,0.0004,0.0006,0.0367,0.0317,0.2500
1,6m,1.0000,0.0333,0.9506,0.0050,0.0031,0.0013,-0.0003,0.0005,0.0008,0.0383,0.0283,0.5000
2,1y,1.0000,0.0332,0.9350,0.0100,0.0050,0.0018,-0.0003,0.0008,0.0010,0.0432,0.0232,1.0000
3,2y,1.0000,0.0350,0.9044,0.0100,0.0078,0.0042,0.0001,0.0022,0.0021,0.0450,0.0250,2.0000
4,3y,1.0000,0.0366,0.9647,0.0150,0.0096,0.0046,0.0003,0.0024,0.0022,0.0516,0.0216,3.0000


# Calibrating swaption data using SABR model

In [99]:
def calculate_sabr_normal_vol(F, K, T, alpha, rho, nu):
    if F == K:
        # ATM Case (K = F)
        sigma_n = alpha * (1 + ((2 - 3 * rho**2) / 24 * nu**2) * T)
    else:
        # OTM Case (K != F)
        zeta = (nu / alpha) * (F - K)
        
        # x_hat(zeta) calculation
        term = np.sqrt(1 - 2 * rho * zeta + zeta**2)
        x_hat_zeta = np.log((term + zeta - rho) / (1 - rho))
        
        # Hagan Normal Volatility Expansion
        sigma_n = alpha * (zeta / x_hat_zeta) * (1 + ((2 - 3 * rho**2) / 24 * nu**2) * T)
    return sigma_n

def bachelier_payer_price(F, K, T, sigma_n, annuity):
    d = (F - K) / (sigma_n * np.sqrt(T))
    price = annuity * ((F - K) * norm.cdf(d) + sigma_n * np.sqrt(T) * norm.pdf(d))
    return price

def bachelier_receiver_price(F, K, T, sigma_n, annuity):
    d = (F - K) / (sigma_n * np.sqrt(T))
    price = annuity * ((K - F) * norm.cdf(-d) + sigma_n * np.sqrt(T) * norm.pdf(d))
    return price

In [100]:
def calculate_model_prices(expiry, tenor, alpha, rho, nu):
    # Select row of swaption_data where expiry and tenor match
    swaption_data_row = swaption_data[(swaption_data['Expiry'] == expiry) & (swaption_data['Tenor'] == tenor)]

    fwd = swaption_data_row['Forward'].iloc[0]
    expiry_t = swaption_data_row['T_expiry'].iloc[0]
    annuity = swaption_data_row['Annuity'].iloc[0]

    strike_payer = swaption_data_row['Strike_Payer'].iloc[0]
    strike_receiver = swaption_data_row['Strike_Receiver'].iloc[0]
    
    # Calculating model vols
    atm_vol = calculate_sabr_normal_vol(fwd, fwd, expiry_t, alpha, rho, nu)
    payer_vol = calculate_sabr_normal_vol(fwd, strike_payer, expiry_t, alpha, rho, nu)
    receiver_vol = calculate_sabr_normal_vol(fwd, strike_receiver, expiry_t, alpha, rho, nu)

    # Calculating model prices
    atm_straddle_model_price = 2 * annuity * (atm_vol * np.sqrt(expiry_t)/np.sqrt(2*np.pi))
    payer_model_price = bachelier_payer_price(fwd, strike_payer, expiry_t, payer_vol, annuity)
    receiver_model_price = bachelier_receiver_price(fwd, strike_receiver, expiry_t, receiver_vol, annuity)
    
    return np.array([atm_straddle_model_price, payer_model_price, receiver_model_price])

In [101]:
def get_market_prices(expiry, tenor):
    swaption_data_row = swaption_data[(swaption_data['Expiry'] == expiry) & (swaption_data['Tenor'] == tenor)]
    return np.array([swaption_data_row['Straddle'].iloc[0], swaption_data_row['Payer'].iloc[0], swaption_data_row['Receiver'].iloc[0]])

In [102]:
def residuals(params, expiry, tenor):

    model_prices = calculate_model_prices(expiry, tenor, params[0], params[1], params[2])
    market_prices = get_market_prices(expiry, tenor)

    return model_prices - market_prices

In [103]:
# Loop through each row in the swaption_data DataFrame
for index, row in swaption_data.iterrows():
    expiry = row['Expiry']
    tenor = row['Tenor']

    # Initial Guesses
    initial_params = [0.00619, 0.01, 0.5]

    # Bounds: ([min_alpha, min_rho, min_nu], [max_alpha, max_rho, max_nu])
    lower_bounds = [0.0001, -0.999, 0.0001]
    upper_bounds = [0.5, 0.999, 2.0]

    # Execution
    result = least_squares(
        residuals, 
        initial_params, 
        bounds=(lower_bounds, upper_bounds),
        args=(expiry, tenor)
    )

    swaption_data.loc[(swaption_data['Expiry'] == expiry) & (swaption_data['Tenor'] == tenor), ['alpha', 'rho', 'nu']] = result.x

In [104]:
# Save swaption_data to csv
swaption_data.to_csv("../data/processed/swaption_data_with_params.csv", index=False)

# Visualisation

In [114]:
import plotly.graph_objects as go

b. Plot the atm vol surface in 2-dimensions, expiry vs strike. What are your observations?

In [115]:
# Calculating all atm vols
swaption_data['ATM Vol'] = swaption_data.apply(
    lambda row: calculate_sabr_normal_vol(
        F=row['Forward'], 
        K=row['Forward'], # For ATM Vol, Strike = Forward
        T=row['T_expiry'], 
        alpha=row['alpha'], 
        rho=row['rho'], 
        nu=row['nu']
    ), 
    axis=1
)

In [116]:
# 1. Organize your calibrated Alphas (or calculated ATM Vols) into a Pivot Table
# Assuming you have a DataFrame 'df' with columns: Expiry, Tenor, ATM_Vol
surface_data = swaption_data.pivot(index='T_expiry', columns='Tenor', values='ATM Vol')

# 2. Create the 3D Surface
fig = go.Figure(data=[go.Surface(
    z=surface_data.values,
    x=surface_data.columns, # Tenors
    y=surface_data.index,    # Expiries
    colorscale='Viridis',
    colorbar_title='ATM Normal Vol (bps)'
)])

# 3. Style the Layout
fig.update_layout(
    title='ATM Swaption Volatility Surface',
    scene=dict(
        xaxis_title='Swap Tenor (Years)',
        yaxis_title='Option Expiry (Years)',
        zaxis_title='Normal Vol (bps)'
    ),
    width=900, height=800
)

fig.show()

* **Volatility "Hump" in the Front-End:** Notice the sharp peak at the very beginning of the surface (short expiry, short tenor). This is the "hump" often seen in USD markets, representing high uncertainty regarding immediate central bank policy or economic data.

* **Mean Reversion:** As you move toward the 10-year Expiry and 30-year Tenor, the surface slopes downward and flattens out. This reflects the market's belief that while things are volatile now, interest rate variance will eventually stabilize over long horizons.

* **Smoothness and Calibration:** The surface is largely smooth with a clear "ridge" running through it. This suggests your SABR calibration was robust across the entire cube.

* **A "Spike" at the Edge:** There is a noticeable vertical spike at the shortest expiry/tenor corner. While this often represents high front-end volatility, double-check that specific cell (3m 1y or 1y1y) to ensure the optimization didn't hit a local minimum, as it stands out quite a bit from the surrounding "terrain."

c. Plot the rho parameter surface. What are your observations?

In [117]:
# Pivot the calibrated Rho values
rho_pivot = swaption_data.pivot(index='T_expiry', columns='Tenor', values='rho')

fig = go.Figure(data=[go.Surface(
    z=rho_pivot.values,
    x=rho_pivot.columns,
    y=rho_pivot.index,
    colorscale='RdBu', # Red-to-Blue is great for Rho to see pos vs neg
    colorbar_title='SABR Rho'
)])

fig.update_layout(
    title='SABR Rho Parameter Surface (Skew)',
    scene=dict(
        xaxis_title='Swap Tenor',
        yaxis_title='Option Expiry',
        zaxis_title='Rho'
    ),
    width=900, height=800
)
fig.show()

* **Positive vs. Negative Skew Regime**: Notice that the surface is split. For shorter expiries and tenors, the $\rho$ values are negative (indicated by the red dip). As you move toward longer tenors and expiries, the surface climbs above zero into positive territory (the blue plateau).
* **Interpretation**: A negative $\rho$ means volatility increases as rates fall (downward skew), while a positive $\rho$ means volatility increases as rates rise (upward skew).
* **Front-End Skew Sharpness**: There is a very distinct "V-shaped" dip in the short-end of the surface (low Option Expiry, low Swap Tenor). This indicates that the market has a much stronger directional bias (correlation) for near-term rate moves compared to long-term ones.
* **Mean Reversion of Skew**: As the Option Expiry increases toward 10y, the surface becomes much flatter. This suggests that for long-dated options, the market prices in a more symmetric distribution of outcomes, with $\rho$ settling into a stable, slightly positive regime.
* **Tenor Dependence**: The skew is not uniform across swap lengths. For a fixed expiry (e.g., 1y), $\rho$ shifts from negative for a 1y swap to positive for a 30y swap. This reflects different hedging pressures—short-term traders may fear falling rates, while long-term pension or insurance hedgers may be more concerned with rising long-term rates.

d. Plot the nu parameter surface. What are your observations?

In [118]:
# Pivot the calibrated Rho values
nu_pivot = swaption_data.pivot(index='T_expiry', columns='Tenor', values='nu')

fig = go.Figure(data=[go.Surface(
    z=nu_pivot.values,
    x=nu_pivot.columns,
    y=nu_pivot.index,
    colorscale='RdBu', # Red-to-Blue is great for Rho to see pos vs neg
    colorbar_title='SABR Nu'
)])

fig.update_layout(
    title='SABR Nu Parameter Surface (Skew)',
    scene=dict(
        xaxis_title='Swap Tenor',
        yaxis_title='Option Expiry',
        zaxis_title='Nu'
    ),
    width=900, height=800
)
fig.show()

* **Extreme Front-End Vol-of-Vol**: There is a massive peak in the "front-left" corner of the chart (short Option Expiry, short Swap Tenor). This indicates that for very near-term options, the market prices in extremely "fat tails," meaning there is a higher perceived probability of massive interest rate jumps than what a normal distribution would suggest.
* **Rapid Term-Structure Decay**: Notice how quickly the surface drops as Option Expiry moves from 3m toward 2y. This shows that "Vol-of-Vol" mean-reverts quickly; while immediate uncertainty is high, the market expects the distribution of interest rate moves to become more "normal" (less curved) over longer horizons.
* **Tenor Sensitivity**: The curvature is much higher for short Swap Tenors (e.g., 1y, 2y) than for long ones (e.g., 30y). This implies that short-term interest rate instruments have much more volatile "smiles" than long-term ones.
* **Stable Long-End Plateau**: For expiries beyond 5 years, the surface becomes quite flat and stable, settling at a $\nu$ value around 0.6 to 0.8. This represents the long-term baseline for market "smile" curvature.

e. Which point on the surface has the most rho rolldown? Rollup?

In [119]:
# 1. Ensure your data is sorted by Expiry for each Tenor
swaption_data = swaption_data.sort_values(['Tenor', 'T_expiry'])

# 2. Calculate the 'Roll' (change in rho per unit of time)
# We divide by the change in years to get the 'slope'
swaption_data['rho_roll'] = swaption_data.groupby('Tenor')['rho'].diff() / \
                            swaption_data.groupby('Tenor')['T_expiry'].diff()

# 3. Find the extrema
most_rolldown = swaption_data.loc[swaption_data['rho_roll'].idxmin()]
most_rollup = swaption_data.loc[swaption_data['rho_roll'].idxmax()]

print(f"Most Rolldown at: {most_rolldown['T_expiry']} expiry, {most_rolldown['Tenor']} tenor")
print(f"Most Rollup at: {most_rollup['T_expiry']} expiry, {most_rollup['Tenor']} tenor")

Most Rolldown at: 10.0 expiry, 1.0 tenor
Most Rollup at: 2.0 expiry, 1.0 tenor


f. Which point on the surface has the most nu rolldown? Rollup?

In [120]:
# 1. Ensure your data is sorted by Expiry for each Tenor
swaption_data = swaption_data.sort_values(['Tenor', 'T_expiry'])

# 2. Calculate the 'Roll' (change in nu per unit of time)
# We divide by the change in years to get the 'slope'
swaption_data['nu_roll'] = swaption_data.groupby('Tenor')['nu'].diff() / \
                            swaption_data.groupby('Tenor')['T_expiry'].diff()

# 3. Find the extrema
most_rolldown = swaption_data.loc[swaption_data['nu_roll'].idxmin()]
most_rollup = swaption_data.loc[swaption_data['nu_roll'].idxmax()]

print(f"Most Rolldown at: {most_rolldown['T_expiry']} expiry, {most_rolldown['Tenor']} tenor")
print(f"Most Rollup at: {most_rollup['T_expiry']} expiry, {most_rollup['Tenor']} tenor")

Most Rolldown at: 1.0 expiry, 1.0 tenor
Most Rollup at: 0.5 expiry, 1.0 tenor


In [121]:
swaption_data.to_csv("calibrated_sabr_data.csv", index=False)